# Pandas Practice: Wine Quality Data

In [ ]:
# --- Starter code: run this cell first ---
import pandas as pd

# The wine CSV uses semicolons as separators
df = pd.read_csv("../../data/winequality-red.csv", delimiter=";")
df.head()

## Part 1: Exploring the Data

### Challenge 1: Inspect the DataFrame

Use the attributes and methods available on DataFrames to answer the following questions:

- How many rows and columns are in the DataFrame?

- What is the data type of each column?

- How many non-null values are in each column?

- Are all variables continuous, or are any categorical?

- What are the min, mean, max, and median for all numeric columns?

In [ ]:
# Shape gives the (rows, columns) count
print(df.shape)

# info() shows the dtype and non-null count for every column at once
df.info()

# describe() reports min, mean, max, and the 50% value (median) for numeric columns.
# All columns are numeric here: quality is an integer score, so it acts as a
# categorical label even though its type is numeric.
df.describe()

**Explanation**

shape gives row/column counts, info() lists dtypes and non-null counts per column, and describe() gives min/mean/max/median for the numeric columns; quality is numeric in type but behaves like a categorical score.


---

## Part 2: Selecting and Filtering Data

### Challenge 2: Select the first 10 rows of the `chlorides` column.

In [ ]:
# Select the chlorides column, then take the first 10 rows with .head()
df["chlorides"].head(10)

**Explanation**

Selecting the chlorides column returns a Series, and .head(10) takes its first 10 rows in index order. .head() defaults to 5 rows, so we pass 10 explicitly.


### 3: Select the last 10 rows of the `chlorides` column.

In [ ]:
# .tail(10) returns the last 10 rows of the selected column
df["chlorides"].tail(10)

**Explanation**

.tail(10) is the mirror of .head(10), returning the last 10 rows of the chlorides Series. This reads from the end of the index without needing to know the total row count.


### 4: Select rows at indices 264 through 282 of the `chlorides` and `density` columns.

In [ ]:
# .loc[] selects by label. The default index here is integer labels 0..1598,
# and .loc is inclusive of the end label, so 264:282 returns rows 264 through 282.
df.loc[264:282, ["chlorides", "density"]]

**Explanation**

.loc[] selects by label, and the default integer index here doubles as the labels. Note .loc is inclusive on both ends, so 264:282 returns row 282 too (unlike .iloc and plain Python slicing). Passing a list of column names keeps just chlorides and density.


### 5: Return all rows where the `chlorides` value is less than 0.10.

In [ ]:
# A boolean mask keeps only the rows where the condition is True
df[df["chlorides"] < 0.10]

**Explanation**

The comparison df['chlorides'] < 0.10 produces a boolean Series, and indexing the DataFrame with it keeps only the True rows. This boolean-mask pattern is the standard way to filter in pandas.


### 6: Return all rows where `chlorides` is above the column mean.

> Tip: don't hard-code the mean value, use a method to calculate it.

In [ ]:
# Compute the mean with .mean() instead of hard-coding a number, so the filter
# stays correct even if the data changes
mean_chlorides = df["chlorides"].mean()
df[df["chlorides"] > mean_chlorides]

**Explanation**

We call .mean() to compute the average instead of typing a literal, so the filter stays correct if the data changes. The boolean mask then keeps rows above that computed mean.


### 7: Return all rows where `pH` is greater than 3.0 and less than 3.5.

In [ ]:
# Combine two conditions with & (bitwise AND). Each condition must be wrapped
# in parentheses because & binds tighter than the comparison operators.
ph_filtered = df[(df["pH"] > 3.0) & (df["pH"] < 3.5)]
ph_filtered

**Explanation**

Two conditions are combined with & (element-wise AND), and each must be wrapped in parentheses because & has higher precedence than the comparison operators. The result is stored in ph_filtered so Challenge 8 can build on it.


### 8: From the result in Challenge 7, further filter to keep only rows where `residual sugar` is less than 2.0.

In [ ]:
# Take the result from Challenge 7 and apply a second mask on top of it
ph_filtered[ph_filtered["residual sugar"] < 2.0]

**Explanation**

This reuses ph_filtered from Challenge 7 and applies a second boolean mask on top, narrowing to rows with residual sugar below 2.0. Chaining filters this way is equivalent to combining all conditions in one mask.


---

## Part 3: Aggregating and Transforming Data

### 9: Find the average amount of `chlorides` for each `quality` value.

In [ ]:
# Group rows by quality, then take the mean of chlorides within each group
df.groupby("quality")["chlorides"].mean()

**Explanation**

groupby('quality') splits rows into one group per quality score, and ['chlorides'].mean() averages chlorides within each group. The result is a Series indexed by quality.


### 10: For wines where `pH` is between 3.0 and 4.0 (exclusive), find the average `alcohol` value grouped by `pH`.

In [ ]:
# First filter to the exclusive pH range, then group by pH and average alcohol
ph_range = df[(df["pH"] > 3.0) & (df["pH"] < 4.0)]
ph_range.groupby("pH")["alcohol"].mean()

**Explanation**

First we filter to the exclusive pH range (greater than 3.0 and less than 4.0), then group the remaining rows by pH and average alcohol. Grouping on a float column like pH yields one group per distinct value.


### 11: For wines where `alcohol` is between 9.25 and 9.5, find the highest `residual sugar` value.

In [ ]:
# Filter to the alcohol band, then take the max of residual sugar in that subset
alcohol_band = df[(df["alcohol"] >= 9.25) & (df["alcohol"] <= 9.5)]
alcohol_band["residual sugar"].max()

**Explanation**

We filter to the alcohol band between 9.25 and 9.5 (inclusive here via >= and <=), then take .max() of residual sugar within that subset. A single .max() call returns one scalar value.


### 12: Create a new column called `total_acidity` that is the sum of `fixed acidity` and `volatile acidity`.

In [ ]:
# Adding two columns aligns by row, so each row gets its own summed value
df["total_acidity"] = df["fixed acidity"] + df["volatile acidity"]
df.head()

**Explanation**

Adding two columns with + aligns them row by row, so each new total_acidity value is that row's fixed plus volatile acidity. Assigning to df['total_acidity'] adds the column in place.


### 13: Find the average `total_acidity` for each `quality` value.

In [ ]:
# Uses the total_acidity column created in Challenge 12, grouped by quality
df.groupby("quality")["total_acidity"].mean()

**Explanation**

This relies on the total_acidity column created in Challenge 12, so that cell must run first. We group by quality and average total_acidity within each group.


### 14: Find the 5 highest `density` values.

In [ ]:
# nlargest is a concise way to get the top n values without sorting the whole frame
df["density"].nlargest(5)

**Explanation**

.nlargest(5) returns the 5 highest density values already sorted descending, which is clearer and faster than sorting the whole column and slicing. It operates on the Series, so the output is just the density values.


### 15: Find the 10 lowest `sulphates` values.

In [ ]:
# nsmallest returns the n lowest values, the mirror of nlargest
df["sulphates"].nsmallest(10)

**Explanation**

.nsmallest(10) is the counterpart to nlargest and returns the 10 lowest sulphates values sorted ascending. It avoids a full sort of the column to get just the smallest values.
